# Notebook 01 — Bronze Ingestion

**Project:** End-to-End Retail Data Engineering Pipeline (Olist)

## Objective
Load the raw Olist CSV batches that Azure Data Factory copied into ADLS Gen2 (exposed through the lakehouse shortcut) into **Bronze Delta tables**.

## Design rules
- Bronze keeps source values **exactly as delivered**: every column is a string, nothing is filtered, cleaned or de-duplicated.
- Three lineage columns are added: `_source_file`, `_batch_date`, `_ingested_at`.
- Each run is a **full refresh** (overwrite) of every Bronze table. Incremental loading is *not* implemented in this project.
- Ingestion is validated against `manifest.json`, which the source-simulation script wrote next to the data.

## Source / destination
`Files/retail/source/<table>/batch_date=YYYY-MM-DD/<table>.csv` → managed Delta tables `bronze_<table>`

## 1. Configuration

Define the source location, the nine source tables, and the CSV read options.

Only `order_reviews` contains quoted free-text comments with embedded line breaks and commas, so only that table needs `multiLine` parsing. Reading it without these options splits one review across several rows (this was the 105,225-row problem found earlier).

In [1]:
from pyspark.sql import functions as F

SOURCE_PATH = "Files/retail/source"

TABLES = [
    "orders", "customers", "order_items", "order_payments", "order_reviews",
    "products", "sellers", "geolocation", "category_translation",
]

# Every column is read as a string (inferSchema=false); types are applied in Silver.
BASE_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
    "recursiveFileLookup": "true",   # find <table>/batch_date=*/<table>.csv
    "pathGlobFilter": "*.csv",
}
REVIEW_OPTIONS = {"multiLine": "true", "quote": '"', "escape": '"'}

StatementMeta(, 5074d373-b78e-43a9-bf4a-cd6960614183, 3, Finished, Available, Finished, False)

## 2. Read the Source Manifest

The manifest records, for every table, how many rows and how many batch files the source simulator wrote. It is our independent reference for validating Bronze.

Expected output: one row per table (9 rows).

In [2]:
manifest = (
    spark.read.option("multiLine", "true").json(f"{SOURCE_PATH}/manifest.json")
    .first().asDict(recursive=True)["tables"]
)

display(spark.createDataFrame(
    [(t, manifest[t]["batches"], manifest[t]["bronze_rows_written"]) for t in TABLES],
    ["table_name", "expected_batch_files", "expected_bronze_rows"],
))

StatementMeta(, 5074d373-b78e-43a9-bf4a-cd6960614183, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 94d8cb61-5f14-43a3-8a74-41e99ac7a82d)

## 3. Ingestion Function

`ingest_bronze(table)` reads every monthly batch of one table, adds the lineage columns, and overwrites `bronze_<table>`.

- `_source_file` — full path of the file the row came from.
- `_batch_date` — month-end batch date parsed from the folder name (`batch_date=YYYY-MM-DD`).
- `_ingested_at` — time of this load.

In [3]:
def ingest_bronze(table):
    options = {**BASE_OPTIONS, **(REVIEW_OPTIONS if table == "order_reviews" else {})}

    df = (
        spark.read.options(**options).csv(f"{SOURCE_PATH}/{table}")
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_batch_date", F.regexp_extract("_source_file", r"batch_date=(\d{4}-\d{2}-\d{2})", 1))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(f"bronze_{table}"))

StatementMeta(, 5074d373-b78e-43a9-bf4a-cd6960614183, 5, Finished, Available, Finished, False)

## 4. Load All Bronze Tables

Run the ingestion for all nine tables. This overwrites the Bronze tables, which is safe because Bronze is rebuilt from the immutable source files each time.

Expected output: no errors (a few minutes in total; geolocation is the largest table).

In [4]:
for table in TABLES:
    ingest_bronze(table)
    print(f"loaded bronze_{table}")

StatementMeta(, 5074d373-b78e-43a9-bf4a-cd6960614183, 6, Finished, Available, Finished, False)

loaded bronze_orders
loaded bronze_customers
loaded bronze_order_items
loaded bronze_order_payments
loaded bronze_order_reviews
loaded bronze_products
loaded bronze_sellers
loaded bronze_geolocation
loaded bronze_category_translation


## 5. Validate Bronze Against the Manifest

For every table compare:
- Bronze **row count** with `bronze_rows_written` from the manifest,
- number of distinct **source files** with the manifest's batch count,
- rows with an **unparseable batch date** (must be 0).

Any `MISMATCH` means ingestion is incomplete or parsing is wrong — fix it before moving on to Silver.

In [5]:
rows = []
for table in TABLES:
    b = spark.table(f"bronze_{table}")
    rows.append((
        table,
        manifest[table]["bronze_rows_written"], b.count(),
        manifest[table]["batches"], b.select("_source_file").distinct().count(),
        b.filter(F.col("_batch_date") == "").count(),
    ))

check = (
    spark.createDataFrame(rows, ["table_name", "expected_rows", "bronze_rows",
                                 "expected_files", "bronze_files", "missing_batch_date"])
    .withColumn("status", F.when(
        (F.col("expected_rows") == F.col("bronze_rows"))
        & (F.col("expected_files") == F.col("bronze_files"))
        & (F.col("missing_batch_date") == 0), "OK").otherwise("MISMATCH"))
)
display(check.orderBy("table_name"))

assert check.filter("status != 'OK'").count() == 0, "Bronze does not match the source manifest - investigate before Silver."
print("Bronze layer validated: all 9 tables match the manifest.")

StatementMeta(, 5074d373-b78e-43a9-bf4a-cd6960614183, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2feb5507-0a7a-4b06-8041-ab59681e7fd7)

Bronze layer validated: all 9 tables match the manifest.


## 6. Summary

| Item | Result |
|---|---|
| Bronze tables | 9 managed Delta tables (`bronze_*`) |
| Load type | Full refresh, source values preserved as strings |
| Lineage | `_source_file`, `_batch_date`, `_ingested_at` |
| Validation | Row counts and file counts match `manifest.json` |

Duplicates, invalid records and superseded order versions are **intentionally still present** in Bronze. They are handled in Notebook 02 (Silver).